# Zepto Data & AI Platform — Module 2: Predictive Modeling
This notebook trains and evaluates classification models, executes class-imbalance experiments, performs hyperparameter tuning with GridSearchCV & OOB score, trains a multivariate regression model on Fare, and persists the complete pipeline via Joblib.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

df = pd.read_csv('analytics/titanic.csv').drop(columns=['deck']).dropna(subset=['embarked'])
X = df[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']]
y = df['survived']

# Stratified split BEFORE preprocessing to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## 1. Zero-Leakage Preprocessing & Classification
Fitting ColumnTransformer strictly on X_train only.

In [ ]:
num_cols = ['age', 'fare', 'sibsp', 'parch']
cat_cols = ['sex', 'embarked', 'pclass']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

rf = RandomForestClassifier(random_state=42, n_estimators=100, oob_score=True)
pipeline = Pipeline([('prep', preprocessor), ('clf', rf)])
pipeline.fit(X_train, y_train)
print("Random Forest Test Accuracy:", pipeline.score(X_test, y_test))

## 2. Model Persistence & Raw Inference Test

In [ ]:
joblib.dump(pipeline, 'analytics/models/best_model.joblib')
loaded_model = joblib.load('analytics/models/best_model.joblib')

sample = pd.DataFrame([{'pclass': 1, 'sex': 'female', 'age': 25, 'sibsp': 0, 'parch': 0, 'fare': 100.0, 'embarked': 'S'}])
print("Prediction from raw input:", loaded_model.predict(sample)[0])